## Structure Output 

when we wanted the model to give a structure output 

## Pydantic

In [1]:
import os 
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash")



In [2]:
from pydantic import BaseModel ,  Field 

class Movie(BaseModel):
    title : str = Field(description="This is the title of the movie")
    year : int = Field(description="This is the year the movie was released")
    director : str = Field(description="This is the director of the film")
    rating : float = Field(description="the movies ratings out of 10")

    



In [3]:
model_with_str = model.with_structured_output(Movie)
model_with_str

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x1153c6590>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'description': 'This is the title of the movie', 'title': 'Title', 'type': 'string'

In [4]:
model_with_str.invoke("Provide details of the Movie the passion of christ")

Movie(title='The Passion of the Christ', year=2004, director='Mel Gibson', rating=7.2)

Mesaged output along side parsed structure

In [6]:
from pydantic import BaseModel ,  Field 

class Movie(BaseModel):
    title : str = Field(...,description="This is the title of the movie")
    year : int = Field(...,description="This is the year the movie was released")
    director : str = Field(...,description="This is the director of the film")
    rating : float = Field(...,description="the movies ratings out of 10")

model_with_raw =model.with_structured_output(Movie , include_raw=True)
model_with_raw.invoke("Provide details of the Movie the passion of christ")


{'raw': AIMessage(content='{"title":"The Passion of the Christ","year":2004,"director":"Mel Gibson","rating":7.2}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e774e-ee7b-70f3-aab5-e1d1ab2fa0a3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 179, 'total_tokens': 189, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 152}}),
 'parsed': Movie(title='The Passion of the Christ', year=2004, director='Mel Gibson', rating=7.2),
 'parsing_error': None}

# Nested Structure

In [7]:
from pydantic import BaseModel , Field

class Actor(BaseModel):
    actor : str 
    role : str 

class MovieDetails(Actor):
    title : str
    year : str 
    cast : list[Actor]
    genre : list[str]
    budget : float | None = Field(None , description= "Budget in million INR")

In [8]:
model_with_details = model.with_structured_output(MovieDetails)
model_with_details.invoke("The Series Chosen")

MovieDetails(actor='Saif Ali Khan', role='Sartaj Singh', title='Sacred Games', year='2018', cast=[Actor(actor='Saif Ali Khan', role='Sartaj Singh'), Actor(actor='Nawazuddin Siddiqui', role='Ganesh Gaitonde'), Actor(actor='Radhika Apte', role='Anjali Mathur')], genre=['Crime', 'Thriller', 'Drama'], budget=45.0)

## TypedDict

Run time validation is not there in this 

In [15]:
from typing_extensions import TypedDict , Annotated

class Movie(TypedDict):
    title : Annotated[str, ... ,"Tile of the movie"]
    Actor : Annotated[str, ... ,"Actor of the movie"]
    director : Annotated[str, ... ,"director of the movie"]
    rating : Annotated[float, ... ,"Rating of the movie"]

model_with_tyd= model.with_structured_output(Movie)
model_with_tyd.invoke("Provide the details of the movie Baahubali")

{'title': 'Baahubali: The Beginning',
 'Actor': 'Prabhas',
 'director': 'S. S. Rajamouli',
 'rating': 8.0}

In [19]:
from typing_extensions import TypedDict , Annotated

class Actor(TypedDict):
    actor : str 
    role : str 

class MovieDetails(TypedDict):
    title : str
    year : str 
    cast : list[Actor]
    genre : list[str]
    budget : float | None = Field(None , description= "Budget in million INR")

model_with_details = model.with_structured_output(MovieDetails)
model_with_details.invoke("The Series Chosen")

{'title': 'Game of Thrones',
 'year': '2011-2019',
 'cast': [{'actor': 'Peter Dinklage', 'role': 'Tyrion Lannister'},
  {'actor': 'Lena Headey', 'role': 'Cersei Lannister'},
  {'actor': 'Kit Harington', 'role': 'Jon Snow'},
  {'actor': 'Emilia Clarke', 'role': 'Daenerys Targaryen'}],
 'genre': ['Fantasy', 'Drama', 'Adventure'],
 'budget': 1000000000}

In [20]:
model.profile

{'name': 'Gemini 2.5 Flash',
 'release_date': '2025-03-20',
 'last_updated': '2025-06-05',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

## Data Classes

These are nothing but classes which contains data . typically not a restriction  , but can be used via @dataclass decorators

In [26]:
from langchain.agents import create_agent
from pydantic import BaseModel , Field

class ContactDetails(BaseModel):
    name: str = Field(description="Name of the person")
    phone_number: int = Field(description="Phno of the person")

agent = create_agent(model = "google_genai:gemini-2.5-flash" , 
                     response_format= ContactDetails)


result= agent.invoke({"messages" :[{"role": "user" , "content": "Provide the details of Ravi whose phone number is 67895432990"}]})

result["structured_response"]



ContactDetails(name='Ravi', phone_number=67895432990)

In [27]:
# DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactDetails():
    name: str 
    phone_number: int 

agent = create_agent(model = "google_genai:gemini-2.5-flash" , 
                     response_format= ContactDetails)


result= agent.invoke({"messages" :[{"role": "user" , "content": "Provide the details of Ravi whose phone number is 67895432990"}]})

result["structured_response"]


ContactDetails(name='Ravi', phone_number=67895432990)